# Module 2 — Part 4: Generate Module 3 Handoff File

**Objective:** Load both trained models (classification from Part 2, regression from Part 3), generate delay predictions on the test set, and combine them with cost/timing/business columns into a single file for Module 3 (Prescriptive Optimization Engine).

**Input files:**
- `X_train.csv`, `X_test.csv` — original data, for business context columns
- `delay_classification_xgb.json` + `classification_preprocessing.pkl` (from Part 2)
- `delay_duration_xgb.json` + `regression_preprocessing.pkl` (from Part 3)


## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import xgboost as xgb

pd.set_option('display.max_columns', None)


## 2. Load Original Data and Saved Models

In [2]:
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')

with open('classification_preprocessing.pkl', 'rb') as f:
    clf_artifacts = pickle.load(f)
with open('regression_preprocessing.pkl', 'rb') as f:
    reg_artifacts = pickle.load(f)

clf_model = xgb.XGBClassifier()
clf_model.load_model('delay_classification_xgb.json')

reg_model = xgb.XGBRegressor()
reg_model.load_model('delay_duration_xgb.json')

print("Models and preprocessing artifacts loaded.")


Models and preprocessing artifacts loaded.


## 3. Reproduce Preprocessing

Reapplies the exact frequency/one-hot encoding pipeline used at training time, fit on `X_train` only.


In [3]:
def apply_preprocessing(X_raw, artifacts, X_train_reference):
    X_p = X_raw.drop(columns=artifacts['drop_cols']).copy()
    for col in artifacts['high_card_cols']:
        freq_map = X_train_reference[col].value_counts()
        X_p[col + '_freq'] = X_p[col].map(freq_map).fillna(0)
    X_p = X_p.drop(columns=artifacts['high_card_cols'])
    X_p = pd.get_dummies(X_p, columns=artifacts['low_card_cols'], drop_first=True)
    X_p = X_p.reindex(columns=artifacts['final_columns'], fill_value=0)
    return X_p

X_test_clf_ready = apply_preprocessing(X_test, clf_artifacts, X_train)
X_test_reg_ready = apply_preprocessing(X_test, reg_artifacts, X_train)
print("Preprocessing applied to X_test.")


Preprocessing applied to X_test.


## 4. Generate Predictions

In [4]:
delay_probability = clf_model.predict_proba(X_test_clf_ready)[:, 1]
predicted_delay_duration = np.clip(reg_model.predict(X_test_reg_ready), 0, None)

print(f"Generated {len(delay_probability)} predictions.")


Generated 36104 predictions.


## 5. Combine with Business Context and Save

Adds cost, timing, and business columns Module 3 needs to build its optimization objective and constraints.


In [5]:
context_cols = [
    'order_item_total', 'sales', 'product_price', 'order_item_quantity',
    'benefit_per_order', 'order_profit_per_order', 'order_item_discount',
    'shipping_mode', 'days_for_shipment_scheduled', 'order_region',
    'order_country', 'category_name', 'customer_segment', 'market', 'order_status'
]

module3_output = X_test[context_cols].copy()
module3_output.insert(0, 'shipment_row_id', X_test.index)
module3_output['delay_probability'] = delay_probability.round(4)
module3_output['predicted_delay_duration_days'] = predicted_delay_duration.round(2)
module3_output['is_high_risk'] = (module3_output['delay_probability'] >= 0.5).astype(int)

module3_output.to_csv('module3_input_predictions.csv', index=False)
print(f"Saved module3_input_predictions.csv — shape: {module3_output.shape}")
module3_output.head()


Saved module3_input_predictions.csv — shape: (36104, 19)


,shipment_row_id,order_item_total,sales,product_price,order_item_quantity,benefit_per_order,order_profit_per_order,order_item_discount,shipping_mode,days_for_shipment_scheduled,order_region,order_country,category_name,customer_segment,market,order_status,delay_probability,predicted_delay_duration_days,is_high_risk
0,0,199.990005,199.990005,199.990005,1,100.000000,100.000000,0.00,Standard Class,4,West of USA,Estados Unidos,Water Sports,Corporate,USCA,CLOSED,0.2373,0.51,0
1,1,254.979996,299.980011,299.980011,1,86.690002,86.690002,45.00,Standard Class,4,Central America,Guatemala,Camping & Hiking,Consumer,LATAM,PENDING_PAYMENT,0.4077,0.66,0
2,2,111.580002,119.980003,59.990002,2,54.669998,54.669998,8.40,Standard Class,4,Northern Europe,Reino Unido,Cleats,Consumer,Europe,PROCESSING,0.3437,0.60,0
3,3,395.980011,399.980011,399.980011,1,123.940002,123.940002,4.00,Same Day,0,East of USA,Estados Unidos,Fishing,Consumer,USCA,CLOSED,0.2122,0.08,0
4,4,185.929993,199.919998,49.980000,4,58.189999,58.189999,13.99,Same Day,0,Western Europe,Francia,Indoor/Outdoor Games,Consumer,Europe,CLOSED,0.6781,0.86,1


**Note on `shipment_row_id`:** the Module 1 dataset has no unique order/shipment ID column, so this uses the DataFrame row index as a stand-in. Flag this with the team — Module 4's database write-back will eventually need a real order identifier.


## Summary

- Loaded both trained models (classification + regression) from Parts 2 and 3
- Reproduced the exact preprocessing pipeline for each
- Generated per-shipment delay probability and expected delay duration
- Combined predictions with cost/timing/business context columns
- Saved `module3_input_predictions.csv` — the direct handoff file for Module 3

**Module 2 is now complete.** Together, Parts 1–4 answer "What will happen?" — predicting both delay probability and expected delay duration — which feeds directly into Module 3's optimization engine.
